# Context is the Asset
## SAP HANA Cloud Vector Engine · Claude API · Vantara Capital Group

**Full pipeline** — context store → similarity search → lifecycle → context package → LLM answer

Context Engineering with SAP HANA Cloud Vector Engine — Building the AI Memory Layer

---

## Cell 1 — Install dependencies

In [ ]:
!pip install hdbcli anthropic --quiet
print('Dependencies ready.')

## Cell 2 — Configuration
Fill in your SAP HANA Cloud credentials and Anthropic API key.

In [ ]:
# ── SAP HANA Cloud ─────────────────────────────────────────────
HANA_HOST     = 'YOUR_HOST.hanacloud.ondemand.com'
HANA_PORT     = 443
HANA_USER     = 'YOUR_USER'
HANA_PASSWORD = 'YOUR_PASSWORD'

# ── Anthropic ──────────────────────────────────────────────────
ANTHROPIC_API_KEY = 'sk-ant-...'

# ── Pipeline parameters ────────────────────────────────────────
TABLE_NAME      = 'FIN_CONTEXT_VANTARA'
EMBEDDING_MODEL = 'SAP_NEB.20240715'
TOP_K           = 3
CLAUDE_MODEL    = 'claude-sonnet-4-20250514'
TEMPERATURE     = 0.2
MAX_TOKENS      = 1024

print(f'Config loaded.')
print(f'  HANA    : {HANA_HOST}')
print(f'  Model   : {EMBEDDING_MODEL}')
print(f'  LLM     : {CLAUDE_MODEL}')
print(f'  Top K   : {TOP_K}')

## Cell 3 — Connect to HANA and verify the context store

In [ ]:
import hdbcli.dbapi

def get_connection():
    return hdbcli.dbapi.connect(
        address  = HANA_HOST,
        port     = HANA_PORT,
        user     = HANA_USER,
        password = HANA_PASSWORD,
        encrypt  = True
    )

conn   = get_connection()
cursor = conn.cursor()

# Total chunks
cursor.execute(f'SELECT COUNT(*) FROM "{TABLE_NAME}"')
total = cursor.fetchone()[0]

# Active chunks
cursor.execute(f"""
    SELECT COUNT(*) FROM "{TABLE_NAME}"
    WHERE "IS_ACTIVE" = 'Y' AND "EXPIRY_DATE" > CURRENT_DATE
""")
active = cursor.fetchone()[0]

# Breakdown by document
cursor.execute(f"""
    SELECT "DOC_ID", "DOC_VERSION", "FILING_TYPE", COUNT(*) AS CNT
    FROM "{TABLE_NAME}"
    GROUP BY "DOC_ID", "DOC_VERSION", "FILING_TYPE"
    ORDER BY "DOC_ID"
""")
rows = cursor.fetchall()

cursor.close()
conn.close()

print(f'Connected to SAP HANA Cloud.')
print(f'Total chunks  : {total}')
print(f'Active chunks : {active}')
print(f'\nContext store breakdown:')
print(f'  {"DOC_ID":<30} {"VERSION":<10} {"TYPE":<20} {"CHUNKS"}')
print(f'  {"-"*30} {"-"*10} {"-"*20} {"-"*6}')
for r in rows:
    print(f'  {r[0]:<30} {r[1]:<10} {r[2]:<20} {r[3]}')

## Cell 4 — Similarity search
The question is embedded inline using `VECTOR_EMBEDDING()` with type `QUERY`.
Cosine similarity ranks all active, non-expired chunks by semantic relevance.
The `WHERE` clause is the governance layer:
- `IS_ACTIVE = 'Y'` — excludes soft-retired chunks
- `EXPIRY_DATE > CURRENT_DATE` — excludes expired context

In [ ]:
def retrieve_context(question, top_k=None, filing_filter=None):
    k    = top_k or TOP_K
    conn = get_connection()
    cur  = conn.cursor()

    where_extra = ''
    if filing_filter:
        where_extra = f" AND \"FILING_TYPE\" = '{filing_filter}'"

    sql = f"""
        SELECT TOP {k}
            "DOC_ID", "DOC_VERSION", "FILING_TYPE",
            "FISCAL_YEAR", "SECTION", "CHUNK_TEXT",
            COSINE_SIMILARITY(
                "CHUNK_VECTOR",
                VECTOR_EMBEDDING(?, 'QUERY', '{EMBEDDING_MODEL}')
            ) AS "SCORE"
        FROM "{TABLE_NAME}"
        WHERE "IS_ACTIVE" = 'Y'
        AND   "EXPIRY_DATE" > CURRENT_DATE
        {where_extra}
        ORDER BY "SCORE" DESC
    """
    cur.execute(sql, (question,))
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    cur.close()
    conn.close()
    return [dict(zip(cols, r)) for r in rows]


# ── Run the search ─────────────────────────────────────────────
QUESTION = 'Is our commercial real estate concentration within policy limits?'

results = retrieve_context(QUESTION)

print(f'Question: {QUESTION}')
print(f'Retrieved {len(results)} chunks\n')
print(f'  {"Rank":<5} {"Score":<8} {"Doc":<25} {"Ver":<6} {"Section"}')
print(f'  {"-"*5} {"-"*8} {"-"*25} {"-"*6} {"-"*30}')
for i, r in enumerate(results, 1):
    print(f'  {i:<5} {r["SCORE"]:.6f} {r["DOC_ID"][:25]:<25} {r["DOC_VERSION"]:<6} {r["SECTION"]}')

print(f'\nTop chunk preview:')
print(f'  {results[0]["CHUNK_TEXT"][:200]}...')

Cell 5 REMOVED

## Cell 6 — Assemble the context package
Retrieved chunks are assembled into a structured context string with full source attribution:
DOC_ID, version, filing type, fiscal year, section, and similarity score.
The LLM receives this attribution alongside the text and cites it when answering.

In [ ]:
def assemble_context(chunks):
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(
            f"[Source {i}: {c['DOC_ID']} v{c['DOC_VERSION']} | "
            f"{c['FILING_TYPE']} FY{c['FISCAL_YEAR']} | "
            f"{c['SECTION']} | Relevance: {c['SCORE']:.4f}]\n"
            f"{c['CHUNK_TEXT']}"
        )
    return '\n\n---\n\n'.join(parts)


context_package = assemble_context(results)

print('Context package assembled.')
print(f'Chunks    : {len(results)}')
print(f'Characters: {len(context_package):,}')
print(f'Est tokens: ~{len(context_package)//4:,}')
print(f'\n{"-"*60}')
print(context_package)
print(f'{"-"*60}')

## Cell 7 — Send context package to Claude
The LLM is interchangeable. What does not change is the context layer in SAP HANA Cloud.
The system prompt instructs Claude to answer only from the provided context and cite every source.

In [ ]:
import anthropic

def ask_claude(question, context, system_prompt=None):
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    sys_p  = system_prompt or (
        'You are a portfolio risk analyst assistant for Vantara Capital Group. '
        'Answer using ONLY the provided context. '
        'For every factual claim cite the source using this exact format: '
        '[Source N: DOC_ID vVERSION | SECTION]. '
        'The document ID, version, and section are in the header of each source block. '
        'If the context does not contain sufficient information, state that clearly. '
        'Do not use your training data for factual claims.'
    )
    response = client.messages.create(
        model      = CLAUDE_MODEL,
        max_tokens = MAX_TOKENS,
        system     = sys_p,
        messages   = [{'role': 'user', 'content': f'CONTEXT:\n{context}\n\nQUESTION:\n{question}'}]
    )
    return response.content[0].text

print(f'Sending to {CLAUDE_MODEL}...')
answer = ask_claude(QUESTION, context_package)
print('Done.')


## Cell 8 — Pipeline summary

In [ ]:
SEP  = chr(9472) * 64
SEP2 = chr(183)  * 64

print(SEP)
print('QUESTION')
print(SEP)
print(f'  {QUESTION}')

print()
print(SEP)
print('RETRIEVED CHUNKS')
print(SEP)
for i, r in enumerate(results, 1):
    score     = r["SCORE"]
    score_bar = chr(9608) * int(score * 20) + chr(9617) * (20 - int(score * 20))
    relevance = "HIGH" if score > 0.75 else "MEDIUM" if score > 0.55 else "LOW"
    print(f"  Rank {i}  [{score_bar}]  {score:.4f}  {relevance}")
    print(f"         {r['DOC_ID']} {r['DOC_VERSION']} · {r['FILING_TYPE']} · {r['SECTION']}")
    print(f"         {r['CHUNK_TEXT'][:120]}...")
    if i < len(results):
        print(f"  {SEP2}")

print()
print(SEP)
print("CONTEXT PACKAGE  →  sent to Claude")
print(SEP)
print(context_package)

print()
print(SEP)
print(f"CLAUDE ANSWER  ·  model: {CLAUDE_MODEL}  ·  temp: {TEMPERATURE}")
print(SEP)
print(answer)

print()
print(SEP)
print("PIPELINE SUMMARY")
print(SEP)
print(f"  Context store  : {TABLE_NAME}")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Chunks used    : {len(results)}")
print(f"  Top score      : {results[0]['SCORE']:.4f}")
print(f"  Context chars  : {len(context_package):,}")
print(f"  Est. tokens    : ~{len(context_package)//4:,}")
print(f"  LLM model      : {CLAUDE_MODEL}")
print(SEP)
print()
print("  The LLM is interchangeable.")
print("  The context layer in SAP HANA Cloud is the asset.")
print(SEP)
